# Generic - JavaScript

All 5 JavaScript examples from [docs/generic.md](https://platob.github.io/yggdryl/generic/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

## Shared vocabulary

In [ ]:
const assert = require('node:assert/strict')
const { enums } = require('yggdryl')

assert.ok(enums.dataTypeIds.includes('int64'))
assert.deepEqual(enums.ioModes, ['overwrite', 'append', 'merge', 'readonly', 'random'])

In [ ]:
const assert = require('node:assert/strict')
const { Scalar } = require('yggdryl')

const value = Scalar.fromEnum('io_mode', 'append')
assert.deepEqual([value.enumKind, value.enumValue, value.enumOrdinal], ['io_mode', 'append', 1])
assert.equal(value.asJs(), 'append')

## RecordOptions: every encoding's settings

In [ ]:
const assert = require('node:assert/strict')
const { Field, RecordOptions, fields } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64')], { nullable: false })

const options = RecordOptions.from('trades.parquet')
  .withField(schema)
  .withBatchRowSize(1024)

assert.equal(String(options.mimeType), 'application/vnd.apache.parquet')
assert.equal(options.name, 'row')
assert.ok(options.dtype.equals(schema.dtype))
assert.deepEqual(options.metadata, [])
assert.ok(options.field.equals(schema))
assert.equal(options.batchRowSize, 1024)

// A setting one encoding has reads as null on an encoding that has none.
assert.equal(options.maxRowGroupSize, 1_048_576)
assert.equal(RecordOptions.from('trades.arrows').maxRowGroupSize, null)

In [ ]:
const assert = require('node:assert/strict')
const { Field, RecordOptions, fields } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64')], { nullable: false })
const options = new RecordOptions('trades.arrows')
options.field = schema

// One stored form: declaring only the datatype is the same declaration.
const byDtype = new RecordOptions('trades.arrows').withDtype(schema.dtype)
assert.ok(options.equals(byDtype))
assert.equal(options.stableHash(), byDtype.stableHash())

options.name = 'trade'
assert.equal(options.field.name, 'trade')
assert.ok(options.field.dtype.equals(schema.dtype))

// Entries, a plain object, or a Map declare the metadata alike.
options.metadata = { source: 'exchange' }
assert.deepEqual(options.metadata, [{ key: 'source', value: 'exchange' }])
assert.equal(options.field.get('source'), 'exchange')

// The setter takes a datatype expression as readily as a DataType.
options.dtype = 'struct<id: int64, venue: utf8>'
const built = options.field
assert.equal(built.name, 'trade')
assert.deepEqual([...built.dtype].map((child) => child.name), ['id', 'venue'])
assert.equal(built.get('source'), 'exchange')
assert.equal(built.nullable, false)

// null clears a part; the name stays.
options.dtype = null
options.metadata = []
assert.equal(options.field, null)
assert.deepEqual(options.metadata, [])
assert.equal(options.name, 'trade')

## TypedScalar: one value and its datatype

In [ ]:
const assert = require('node:assert/strict')
const { Scalar } = require('yggdryl')

assert.equal(Scalar.fromJs(42).intoField().name, 'value')
assert.equal(Scalar.fromJs([1, null]).intoArrayField().name, 'item')
assert.equal(Scalar.fromJs([{ id: 1 }]).intoStructField().name, 'row')